# Description of How the Intermediate Representation Works

In this notebook, we will walk you through reading an Intermediate Representation (IR) using the `qret` command.
The IR is the heart of the Quration library, so knowing its structure will help you better understand the library.
This chapter explains intermediate representation while checking how to use the following three subcommands.

- `print`: Print the instruction sequence.
- `diagram`: Create various diagrams related to quantum circuits.
- `simulate`: Simulate a quantum circuit.

The circuit we will examine throughout this chapter is `quration-core/examples/data/circuit/add_craig_5.json`, which we will store in the `example_path` below.

The goals you will reach in this chapter are:

- Use `qret print` to be able to understand the structure of IR in units of functions and instructions.
- Track the positional relationship between `FunctionCall` and `BasicBlock` and be able to explain branching and calling behavior.
- Compare the various diagrams in `diagram` with the results of `simulate`, so that you can consistently understand the structure and check the semantics.

First, we prepare the execution environment and input files.
In the code below, we assume `qret` is build and at its default execution path; otherwise please set the appropriate value according to your environment.

In [ ]:
import pathlib
import os
import platform
import graphviz
from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
example_path = project_root / "quration-core" / "examples" / "data" / "circuit" / "add_craig_5.json"

os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

First, let's verify that `qret` is executable.

In [ ]:
!qret --version

Let's check the input JSON as well.

In [ ]:
Code(filename=example_path, language="json")

## Qret Print

`print` is a subcommand to show the IR instruction sequence.

In this chapter, we will examine the following aspects of `print` in turn:

- Understand the overall picture of the module (number of functions, function names, existence of arguments)
- Order of instructions inside a function (`Call`, measurement, branch, basic instruction)
- How to expand the call hierarchy with optional expansion depth (`-d`)

In [ ]:
!qret print --help

First, let's check the summary of the module with `print -s` to understand which functions are included.

In [ ]:
!qret print -i {example_path} -s

Next, use the `print -s` for a summary of the `UncomputeTemporalAnd` function.

You can check the number of qubits, number of basic blocks, number of instructions, etc.

In [ ]:
!qret print -i {example_path} -s -f "UncomputeTemporalAnd"

### How to Read FunctionCall (Function Call)

Use the `print` subcommand to display the circuit instruction sequence.

In [ ]:
!qret print -i {example_path} -f "UncomputeTemporalAnd"

When a `Call` isntruction appears in the `print` command sequence, it corresponds to a `FunctionCall`.
This representation denotes not just a "call" in terms of control flow, but a structural transition that encompasses both argument binding and return relationships. When reading it, keeping the following two points in mind will make it easier to interpret.

- **Callee function (callee)**: Which function does the program move to?
- **Argument correspondence (operate, etc.)**: How the caller's operand is passed.

For circuits with many reusable functions -- such as `AddCraig(5)` -- following the chain of `Call` instructions first makes it much easier to grasp the big picture without losing track of the high-level function logic. Once you understand the nested calling order, you can naturally align the instruction sequence with the CallGraph shown in the diagram.

### How to Understand a BasicBlock

The `FunctionCall` and `BasicBlock` units are closely related.

A `BasicBlock` is treated as the smallest unit of execution, bounded by branches and jumps.
Therefore, analyzing the circuit in `BasicBlock` units is highly effective for understanding where execution branches and converges -- structures that are difficult to see by looking at the flow of `Call` instructions alone.

When reading the instruction sequence using `print`, following the "reachable nodes" centered around each `BasicBlock` makes it much easier to keep track of the scope of influence of branch conditions.

While the overall flow of `TemporalAnd` is straightforward to follow, `UncomputeTemporalAnd` directly exposes basic block names such as `entry`, `then_0`, and `if_cont_0`.

By mapping these block names to the instruction sequence, you can deepen your understanding to the point of identifying exactly which branch conditions trigger which paths. Even within the same function, this makes it easy to quickly locate where the execution path branches based on measurement results.

In [ ]:
!qret print -i {example_path} -f "TemporalAnd"

Next, we look at `AddCraig(5)` to see how the overall frame of the caller and the expansion depth (`-d`) of internal calls are reflected in the output. First, check the main entry point and call relationships without any expansion, and then use `-d 2` to deepen and trace the internal expansion. Although the number of lines increases as you go deeper recursively, deliberately increasing the depth is highly valuable because it visually highlights exactly where function reuse is occurring.

In [ ]:
!qret print -i {example_path} -f "AddCraig(5)"

Next, we specify `-d 2` to expand and read `FunctionCall` up to two levels deep.
From this output, you can see where nested calls end or are inherited by another function.

In [ ]:
!qret print -i {example_path} -f "AddCraig(5)" -d 2

At this point, you can track the chain of `FunctionCall` and the `BasicBlock` units.
Next, we check the same information in a diagram to visualise the branching and calling paths.

## Qret Diagram

`diagram` is a subcommand to display information about an IR in a diagram.
The following four diagrams are currently implemented.

- Control Flow Graph (CFG)
- Call Graph
- Schematic
- Compute Graph

In [ ]:
!qret diagram --help

### CFG (Control Flow Diagram)

The CFG presents the transitions (branches and continuations) as edges, and the `BasicBlocks` as nodes.
By mapping out the control flow within a `Function` first, you can easily disentangle overlapping execution paths that might otherwise be difficult to follow using `print` alone.

When analyzing the diagram, focus on the following key aspects:

- Identify the entry point (`entry`) and termination points (such as `if` continuations) first.
- Track which blocks are executed next based on the branching conditions (the paths dependent on measurement results).
- Compare whether different reachable nodes exist, even within the same instruction sequence.

In [ ]:
!qret diagram -i {example_path} --function "UncomputeTemporalAnd" --graph-format "CFG" -o { output_dir / "tutorial_2_diagram_cfg.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_cfg.dot")

### CallGraph

A CallGraph is a diagram used to inspect the dependencies of `FunctionCalls` using nodes and edges. For deeply hierarchical circuits like `AddCraig(5)`, identifying which functions are reused via this graph makes it much easier to prioritize downstream optimizations or targets for decomposition.

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "CallGraph" -o { output_dir / "tutorial_2_diagram_call_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_call_graph.dot")

By adding call-count information, you can visualize how many times each function is called, making it easier to identify hot spots and execution paths with high repetition overhead.

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "CallGraph" -o { output_dir / "tutorial_2_diagram_call_graph.dot"} --display_num_calls

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_call_graph.dot")

### ComputeGraph

A ComputeGraph is an auxiliary diagram that visualizes the data dependencies between instructions. It allows you to trace exactly which qubits originated from which operations, how they were modified by subsequent instructions, and where they ultimately contributed. While a `CFG` maps out the path of the control flow, a `ComputeGraph` serves as a blueprint of the underlying data dependencies.

In [ ]:
!qret diagram -i {example_path} --function "TemporalAnd" --graph-format "ComputeGraph" -o { output_dir / "tutorial_2_diagram_compute_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_compute_graph.dot")

### LaTeX

Qret can also generate the diagram in a LaTeX format, which can be imported into reports and papers.

The output uses the [quantikz](https://ctan.org/pkg/quantikz) latex package.

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "LaTeX" -o { output_dir / "tutorial_2_diagram_latex.tex"}

Code(filename=output_dir / "tutorial_2_diagram_latex.tex", language="tex")

## Qret Simulation

`simulate` is a subcommand to simulate the IR.

You can specify how to preserve and simulate the quantum state using the `--state` argument.
There are currently two options supported:

- `Toffoli`: Expresses a quantum state as a list of `(coefficients, calculation basis)`.
    - Set an upper limit on the length of this list with the `--max_superpositions` argument. Default is 1.
    - Suitable for simulating classical circuits such as adder circuits.
    - Has limitations on supported gates (e.g., arbitrary-angle $X$, $Y$, and $Z$ rotations, as well as global phase rotations, are not supported).
- `FullQuantum` : Expresses a $2^n$ quantum state vector of $n$ qubits.
    - Suitable for simulating circuits with a small number of qubits.
    - The multi-controlled X (MCX) gate is not supported.

<small>Note: While both representations are mathematically equivalent, the `Toffoli` simulator consumes significantly less memory because it represents sparse state vectors. By restricting the supported gate set, it keeps the number of active states in superposition limited. If your circuit requires arbitrary-angle rotations ($\theta$), using `FullQuantum` directly is much more efficient.</small>

In [ ]:
!qret simulate --help

In [ ]:
!qret simulate -i {example_path} -f "TemporalAnd" -s Toffoli --max_superpositions 2 --init_state "011"

In [ ]:
!qret simulate -i {example_path} -f "TemporalAnd" -s FullQuantum --init_state "011"

Finally, run `AddCraig(5)` in `Toffoli` to confirm that the addition executes correctly.

The following command initializes the qubit of `dst` to `01100`, initializes the qubit of `src` to `11010`, and performs in-place addition.
After the execution, only the `dst` qubit is expected to change to `10001`.

In [ ]:
!qret simulate -i {example_path} -f "AddCraig(5)" -s Toffoli --max_superpositions 16 --init_state "0110011010000"

### (Advanced) Description of Quantum Circuits According to Classical Probability Distribution

In Quration's intermediate representation (IR), the `DISCRETE_DISTRIBUTION` instruction allows you to instantiate a random integer key into a set of classical registers. You can then pair this with a subsequent `SWITCH` instruction to branch into paths corresponding to that integer key.

In this example, we construct a branching circuit that applies an `X` gate to one of the qubits (`q0`, `q1`, or `q2`) with a uniform $1/3$ probability.

1. How `DISCRETE_DISTRIBUTION` Behaves
- **Integer Selection**: It selects a single integer value between $0$ and $weights.size() - 1$, weighted according to the probabilities defined in the `weights` array.
- **Register Writing**: It writes this selected integer as a bitstring to the classical registers specified in `registers`.
- **Bit Ordering**: This write operation is **LSB-first** (Least Significant Bit first). For example, `@r0` represents the lowest bit ($2^0$), and `@r1` represents the next bit ($2^1$).

2. How `SWITCH` Interprets Keys
- **Integer Conversion**: The `SWITCH` instruction converts the `registers` array into an integer using `BoolArrayAsInt`. Rather than converting from MSB to LSB, it treats registers[0] as the least significant bit (LSB).
- **Value Range**: In this example, the value is a 2-bit integer capable of representing values from 0 to 3.

The specific mapping between the register states and the branching targets is as follows:
- When `@r0=0`, `@r1=0`: Index $0$ $\rightarrow$ `case_X0`
- When `@r0=1`, `@r1=0`: Index $1$ $\rightarrow$ `case_X1`
- When `@r0=0`, `@r1=1`: Index $2$ $\rightarrow$ `case_X2`
- When `@r0=1`, `@r1=1`: Index $3$ $\rightarrow$ `default` (any other value branches to return)

Each `case` is a single basic block that applies an `X` gate to its corresponding single qubit and then branches to return.

Since weights = $[1, 1, 1]$, the paths `case_X0`, `case_X1`, and `case_X2` are chosen with roughly equal ($1/3$) probability. Because index=3 has no explicitly defined case, it falls through to the default block and exits via return.

If you run `simulate --sample_summary`, you will observe the measurement outcomes `100`, `010`, and `001` in roughly equal proportions. Keeping in mind that the output string format is LSB-first (`q0` `q1` `q2`), you can clearly map these outcomes to the target of the X gate application.

First, let's define the input file as `dist_json` so that we can use the same input in subsequent checks.

In [ ]:
dist_json = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_2.json"

Code(
    filename=dist_json,
    language="json",
)

Check the instruction sequence of `DiscreteDistribution` and `Switch` with this `print` command, such that we can understand the different cases in the explanation above.

In [ ]:
!qret print -i {dist_json} -f "Tutorial2Function"

Next, draw the CFG to understand the flow from `entry` to `case_X0`/`X1`/`X2` and `default`.

In [ ]:
!qret diagram -i {dist_json} --function "Tutorial2Function" --graph-format "CFG" -o { output_dir / "tutorial_2_diagram_cfg.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_cfg.dot")

Finally, check the sample distribution with `simulate` and confirm that the observation probabilities for the three `case` are around 1/3.

In [ ]:
!qret simulate -i {dist_json} -f "Tutorial2Function" -s FullQuantum --init_state "000" -n 10000 --sample_summary

## Summary

- First, check the module size and function list with `print -s` and decide what to investigate.
- Next, use `print -f` to check the instruction sequence based on `FunctionCall`, and increase the expansion depth with `-d` if necessary.
- Understand the branching flow using `diagram --graph-format CFG`, focusing on `BasicBlock`
- Check function call dependencies with `CallGraph` and complement data dependencies with `ComputeGraph`